# DGL-COMP-01 / E01 — 実VCF接続前の異常系検証（合成データ限定） v0.3

**対象:** Windows 11 / Google Colab CPU。追加ライブラリ不要。実患者のVCF・臨床資料を一切取得しません。HFトークンも不要です。

**目的:** E00 v0.2のPASSを受け、実VCFで問題になりやすい「複数ALT」「GT・DP・GQ・AD」「欠損値」「cis/trans位相」「同一遺伝子の2変異」を事前に試します。

**安全境界:** このノートブックで生成する `SYNTHETIC_E01_NOT_FOR_SUBMISSION.csv` は**絶対に公式提出しないこと**。架空の遺伝子名・表現型一致度・集団頻度・変異座標だけを扱います。内部のEPCRは順位テスト用の非校正スコアであり、病因確率ではありません。

実VCFへの適用には、主催者のクラウド利用条件が満たされたことを確認し、実VCFのヘッダー構成・リファレンス・注釈・疾患遺伝学の妥当性を独立に評価する必要があります。

## 1. 作業環境（Colabの一時フォルダのみ）

In [ ]:
import csv, json, math, re
from pathlib import Path
from itertools import combinations
WORK = Path('/content/DGL_E01_SYNTHETIC') if Path('/content').exists() else Path('/tmp/DGL_E01_SYNTHETIC')
WORK.mkdir(parents=True, exist_ok=True)
print('Environment: READY; DATA: SYNTHETIC ONLY; external API: DISABLED')

## 2. 合成VCFの作成

7件の単純な正常レコードだけでなく、複数ALT、位相付きヘテロ、低品質、頻度閾値超過、欠損DP、ホモ接合ALT、参照ホモ接合、欠損GTを含めます。`GENE/CSQ/AF/HPO` は説明用の**架空INFOフィールド**であり、公式VCFに存在する前提にはしません。

In [ ]:
SYNTHETIC_VCF = """##fileformat=VCFv4.2
##reference=GRCh38
##INFO=<ID=GENE,Number=A,Type=String,Description="Synthetic allele-specific gene">
##INFO=<ID=CSQ,Number=A,Type=String,Description="Synthetic consequence">
##INFO=<ID=AF,Number=A,Type=Float,Description="Synthetic population AF, NOT sample allele balance">
##INFO=<ID=HPO,Number=A,Type=Float,Description="Synthetic phenotype match">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Total depth">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allele depth [ref,alt1,...]">
##FORMAT=<ID=PS,Number=1,Type=Integer,Description="Phase set">
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	SYNTHETIC_SAMPLE
chr1	100001	.	A	G	70	PASS	GENE=GENE_A;CSQ=missense_variant;AF=0.0001;HPO=0.85	GT:DP:GQ:AD:PS	0/1:30:80:14,16:.
chr2	200002	.	C	A,T	80	PASS	GENE=GENE_B,GENE_C;CSQ=stop_gained,synonymous_variant;AF=0.00002,0.3;HPO=0.92,0.05	GT:DP:GQ:AD:PS	0/1:38:98:20,18,0:.
chr3	300003	.	G	A	75	PASS	GENE=GENE_D;CSQ=splice_donor_variant;AF=0.00002;HPO=0.95	GT:DP:GQ:AD:PS	1|0:40:99:20,20:100
chr3	300123	.	T	C	72	PASS	GENE=GENE_D;CSQ=missense_variant;AF=0.0001;HPO=0.95	GT:DP:GQ:AD:PS	0|1:35:92:16,19:100
chr4	400001	.	A	G	66	PASS	GENE=GENE_E;CSQ=missense_variant;AF=0.0003;HPO=0.82	GT:DP:GQ:AD:PS	0|1:32:90:16,16:50
chr4	400200	.	T	G	65	PASS	GENE=GENE_E;CSQ=frameshift_variant;AF=0.00002;HPO=0.82	GT:DP:GQ:AD:PS	0|1:40:99:20,20:50
chr5	500005	.	A	C	15	PASS	GENE=GENE_F;CSQ=stop_gained;AF=0.00001;HPO=0.85	GT:DP:GQ:AD:PS	0/1:22:75:12,10:.
chr6	600006	.	C	T	90	PASS	GENE=GENE_G;CSQ=missense_variant;AF=0.15;HPO=0.86	GT:DP:GQ:AD:PS	0/1:30:90:15,15:.
chr7	700007	.	G	T	85	PASS	GENE=GENE_H;CSQ=frameshift_variant;AF=0.00001;HPO=0.90	GT:DP:GQ:AD:PS	0/1:.:70:12,13:.
chr8	800008	.	A	AT	99	PASS	GENE=GENE_I;CSQ=frameshift_variant;AF=0.000001;HPO=0.75	GT:DP:GQ:AD:PS	1/1:38:95:0,38:.
chr9	900009	.	G	A	78	PASS	GENE=GENE_J;CSQ=missense_variant;AF=0.00001;HPO=0.8	GT:DP:GQ:AD:PS	0/0:38:99:38,0:.
chr10	1000010	.	C	G	73	PASS	GENE=GENE_K;CSQ=missense_variant;AF=0.00001;HPO=0.7	GT:DP:GQ:AD:PS	./.:30:.:28,2:.
"""
path = WORK/'E01_SYNTHETIC_ONLY.vcf'
path.write_text(SYNTHETIC_VCF, encoding='utf-8')
assert SYNTHETIC_VCF.count('\nchr')==12
print('Synthetic VCF generated: 12 rows (no patient data).')

## 3. VCFレコードをALT別に分解

複数ALTの場合、`GT=0/1` は **1番目のALTだけ** を示し、2番目のALTは保有していません。`AF` はINFOで指定された集団頻度、`AD` は個人サンプルのアレル深度で、混同しません。GT欠損は「参照ホモ接合」ではありません。

In [ ]:
def parse_info(s):
    out={}
    for item in s.split(';'):
        if '=' in item:
            k,v=item.split('=',1);out[k]=v
        elif item:out[item]=True
    return out

def numeric(v, cls=float):
    return None if v in ('', '.', None) else cls(v)

def iter_synthetic_alleles(p):
    header_found=False
    with Path(p).open(encoding='utf-8') as f:
        for line in f:
            if line.startswith('##'): continue
            if line.startswith('#CHROM'):
                assert len(line.rstrip('\n').split('\t'))==10,'Synthetic example requires one sample'
                header_found=True;continue
            if line.startswith('#') or not line.strip(): continue
            assert header_found,'Missing VCF header'
            chrom,pos,vid,ref,alt_s,qual,flt,infos,fmt,sample=line.rstrip('\n').split('\t')
            alts=alt_s.split(',')
            info=parse_info(infos)
            sample_fields=dict(zip(fmt.split(':'),sample.split(':')))
            gt=sample_fields['GT']
            alleles=[numeric(v,int) for v in re.split(r'[/|]',gt)]
            assert len(alleles)==2,'Synthetic demonstration requires diploid GT'
            phased='|' in gt
            ad=sample_fields.get('AD','.').split(',')
            for i,alt in enumerate(alts,1):
                values={key:info[key].split(',') for key in ('GENE','CSQ','AF','HPO')}
                assert all(len(seq)==len(alts) for seq in values.values()),'Allele-specific INFO mismatch'
                hap=[j for j,a in enumerate(alleles) if a==i]
                yield dict(chrom=chrom,pos=int(pos),ref=ref,alt=alt,alt_index=i,
                    gene=values['GENE'][i-1],consequence=values['CSQ'][i-1],
                    af=numeric(values['AF'][i-1]),hpo=float(values['HPO'][i-1]),
                    qual=numeric(qual),flt=flt,gt=gt,alleles=alleles,
                    hap=hap,phased=phased,ps=numeric(sample_fields.get('PS'),str),
                    dp=numeric(sample_fields.get('DP'),int),gq=numeric(sample_fields.get('GQ'),int),
                    ad_alt=numeric(ad[i],int) if len(ad)>i else None)

alleles=list(iter_synthetic_alleles(path))
assert len(alleles)==13 and sum(x['alt_index']==2 for x in alleles)==1
assert any(x['gt']=='0/1' and x['alt_index']==2 and x['hap']==[] for x in alleles)
print('Record/allele expansion: PASS (12 records => 13 ALT candidates)')
print('Multi-allelic genotype matching: PASS')

## 4. QCを PASS / REVIEW / EXCLUDE に分離

データ欠損（DP/GQなど）は**0に変換して無条件除外しない**よう保留に分離します。以下の閾値（DP≥10、GQ≥20、ALT depth≥3、QUAL≥30、AF≤0.01）は合成演習専用です。実患者のVCFにそのまま適用しません。

In [ ]:
def qc(r):
    if not r['hap']:return 'EXCLUDE','no_target_ALT_GT'
    if r['flt'] not in ('PASS','.'):return 'EXCLUDE','FILTER'
    if r['qual'] is not None and r['qual']<30:return 'EXCLUDE','QUAL'
    if r['af'] is not None and r['af']>0.01:return 'EXCLUDE','AF'
    if r['qual'] is None or r['dp'] is None or r['gq'] is None or r['ad_alt'] is None or r['af'] is None:
        return 'REVIEW','missing_annotation_or_QC'
    if r['dp']<10:return 'EXCLUDE','DP'
    if r['gq']<20:return 'EXCLUDE','GQ'
    if r['ad_alt']<3:return 'EXCLUDE','ALT_DEPTH'
    return 'PASS','ok'

for a in alleles:a['qc'],a['qc_reason']=qc(a)
from collections import Counter
counts=Counter(a['qc'] for a in alleles)
assert counts=={'PASS':7,'EXCLUDE':5,'REVIEW':1},counts
print('QC categorical handling: PASS')
print('PASS:',counts['PASS'],'REVIEW:',counts['REVIEW'],'EXCLUDE:',counts['EXCLUDE'])

## 5. 候補変異をランク付けし、複合ヘテロの位相を評価

**同一phase set（PS）、位相付き2倍体GT、互いに異なるハプロタイプにALTがある場合のみ**、この合成演習ではtrans候補と呼びます。異なるPSや位相がない場合、位相不明のまま保留します。2つの変異があるだけではtransと断定できません。

合成の `hpo` は実HPOオントロジーの類似度ではありません。

In [ ]:
IMPACT={'stop_gained':1.0,'frameshift_variant':.98,'splice_donor_variant':.96,
        'missense_variant':.58,'synonymous_variant':.04}

def score(a):
    # Artificial rank score; NOT calibrated EPCR and NOT a clinical prediction
    rarity=min(1.0,-math.log10(max(a['af'],1e-6))/6)
    return .5*IMPACT[a['consequence']]+.3*a['hpo']+.2*rarity

def phase_relation(a,b):
    if not(a['phased'] and b['phased'] and a['ps'] is not None and a['ps']==b['ps']):
        return 'unknown'
    if len(a['hap'])!=1 or len(b['hap'])!=1:return 'unknown'
    return 'trans' if a['hap'][0]!=b['hap'][0] else 'cis'

passed=[a for a in alleles if a['qc']=='PASS']
trans_pairs=[];cis_pairs=[];uncertain_pairs=[]
for a,b in combinations(passed,2):
    if a['gene'] != b['gene'] or a['chrom'] != b['chrom'] or a['pos']==b['pos']: continue
    if len(a['hap'])!=1 or len(b['hap'])!=1:continue
    rel=phase_relation(a,b)
    if rel=='trans':trans_pairs.append((a,b))
    elif rel=='cis':cis_pairs.append((a,b))
    else:uncertain_pairs.append((a,b))
assert len(trans_pairs)==1 and len(cis_pairs)==1
assert trans_pairs[0][0]['gene']=='GENE_D'
assert cis_pairs[0][0]['gene']=='GENE_E'
print('Phase analysis: PASS (1 trans, 1 cis; synthetic only)')

ranked=[]
for a in passed:
    ranked.append((score(a),a,None))
for a,b in trans_pairs:
    ranked.append((min(0.999,(score(a)+score(b))/2+.07),a,b))
ranked.sort(key=lambda x:(-x[0],x[1]['gene'],x[1]['pos']))
print('Ranked synthetic hypotheses:',len(ranked))

## 6. 公式12列CSV互換の **提出禁止サンプル** を生成

公式テンプレートのヘッダーのみ利用します。架空ID `SYNTHETIC_ONLY` と `DO_NOT_SUBMIT` の注意書きを必ず付けます。合成スコアをEPCR列へ記入しますが、これは**提出上の確率推定ではありません**。

In [ ]:
COLUMNS='proband_id,chrom_1,pos_1,ref_1,alt_1,chrom_2,pos_2,ref_2,alt_2,epcr,finding_type,notes'.split(',')
out=WORK/'SYNTHETIC_E01_NOT_FOR_SUBMISSION.csv'
with out.open('w',newline='',encoding='utf-8') as f:
    w=csv.DictWriter(f,fieldnames=COLUMNS);w.writeheader()
    for sc,a,b in ranked[:10]:
        w.writerow(dict(proband_id='SYNTHETIC_ONLY',chrom_1=a['chrom'],pos_1=a['pos'],
            ref_1=a['ref'],alt_1=a['alt'],chrom_2=b['chrom'] if b else '',
            pos_2=b['pos'] if b else '',ref_2=b['ref'] if b else '',alt_2=b['alt'] if b else '',
            epcr=f'{sc:.6f}',finding_type='primary',notes='DO_NOT_SUBMIT_SYNTHETIC'))
with out.open(newline='',encoding='utf-8') as f:
    r=csv.DictReader(f);assert r.fieldnames==COLUMNS;rows=list(r)
assert 1<=len(rows)<=10
assert all(row['proband_id']=='SYNTHETIC_ONLY' for row in rows)
assert all('DO_NOT_SUBMIT' in row['notes'] for row in rows)
assert all(0<float(row['epcr'])<=1 for row in rows)
assert [float(row['epcr']) for row in rows]==sorted([float(row['epcr']) for row in rows],reverse=True)
assert sum(bool(row['chrom_2']) for row in rows)==1
print('Official 12-column compatibility: PASS (synthetic only)')
print('Compound-het trans pair output: PASS (synthetic only)')
print('Official submission performed: NO')

## 7. 会話へ共有可能な集計結果のみ表示

患者データ取得の準備が整った後も、**実データの変異座標、HPO、サンプルID、遺伝子型、患者臨床記述はこの会話へ貼り付けません**。共有するのはコード上のエラー、患者非依存の仕様、件数等の最小限の安全な集計値だけです。

In [ ]:
summary={
  'experiment':'E01_SYNTHETIC_VCF_EDGECASES_v0.3',
  'input_vcf_records':12,
  'expanded_alt_alleles':len(alleles),
  'qc_pass':counts['PASS'],
  'qc_review':counts['REVIEW'],
  'qc_exclude':counts['EXCLUDE'],
  'synthetic_trans_pairs':len(trans_pairs),
  'synthetic_cis_pairs_not_paired':len(cis_pairs),
  'synthetic_ranked_hypotheses':len(ranked),
  'official_csv_header':'PASS',
  'data':'SYNTHETIC ONLY',
  'official_submission':'NOT PERFORMED',
}
(WORK/'E01_SHAREABLE_SYNTHETIC_SUMMARY.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
for k,v in summary.items():print(f'{k}: {v}')

## 8. DGL-RD / DGL-SYS移植前の注意点

- 実VCFに人工の `GENE/CSQ/HPO/AF` 注釈は必ずしも存在しません。実データの注釈元・品質・Allele Number（A/R/G）の検証を先行させます。
- REF/ALTは必要に応じて正規化とGRCh38参照配列チェックを実施します。多アレルの分解時にGENEやFORMAT Number=G等の配列を機械的に分割する実装は危険です。
- ここでの `phase_relation` は、**同じPSブロック内の信頼できる位相データという条件を仮定**した合成テストです。臨床的なtrans証拠には品質評価が必要です。
- AF欠損は頻度0ではありません。外部ClinVar・gnomAD・HPO情報を実際のバージョン・ライセンスとともに取得する工程が必要です。
- 未校正のスコアを真の「原因確率」として報告しません。公式に必要なEPCRの設定方針は独立に記録します。
- 保存・共有・GitHub公開は**コードと非患者由来の合成実験結果のみ**です。実データは主催者のクラウド利用許可を確認してから扱います。